In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from pyswarm import pso
from PyEMD import EMD
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import copy
import json

# Plot configuration
plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False

# Core configuration
SEQUENCE_LENGTH = 3 
SAVE_DIR = 'non-feature_results' 
EMD_TARGET_COL = 1  
EMD_KEEP_IMFS = None   
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")  
EARLY_STOP_PATIENCE = 20
EARLY_STOP_MIN_DELTA = 1e-6

# Fix random seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def decompose_emd(ts_data):
    emd = EMD() 
    emd.emd(ts_data)
    imfs = emd.imfs
    residue = emd.residue
    
    residue = residue.reshape(1, -1)
    all_emd_components = np.vstack([imfs, residue])
    
    if EMD_KEEP_IMFS is not None and isinstance(EMD_KEEP_IMFS, int):
        all_emd_components = np.vstack([all_emd_components[:EMD_KEEP_IMFS], all_emd_components[-1:]])
    
    emd_features = all_emd_components.T
    return emd_features

def prepare_data_with_emd_unified_feat(file_path):
    data = pd.read_csv(file_path).dropna()
    col2_raw = data.iloc[:, EMD_TARGET_COL].values
    col2_log = np.log(col2_raw)
    
    time_col = data.iloc[:, 0]
    target = col2_log.reshape(-1, 1)
    emd_input = col2_log
    
    emd_features = decompose_emd(emd_input)
    emd_scaler = MinMaxScaler(feature_range=(-1, 1))
    emd_feat = emd_scaler.fit_transform(emd_features)

    target_scaler = MinMaxScaler(feature_range=(-1, 1))
    target_scaled = target_scaler.fit_transform(target)
    
    # 仅保留EMD特征集（移除情感特征相关）
    feature_sets = {"EMD": emd_feat}
    feature_sets_data = {}
    for feat_set_name, feat_data in feature_sets.items():
        n_feats = feat_data.shape[1]
        X, y = [], []
        for i in range(len(feat_data) - SEQUENCE_LENGTH):
            X.append(feat_data[i:i + SEQUENCE_LENGTH])
            y.append(target_scaled[i + SEQUENCE_LENGTH, 0])
        
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, shuffle=False
        )
        
        X_train_tensor = torch.FloatTensor(X_train).to(DEVICE)
        y_train_tensor = torch.FloatTensor(y_train).to(DEVICE)
        X_test_tensor = torch.FloatTensor(X_test).to(DEVICE)
        y_test_tensor = torch.FloatTensor(y_test).to(DEVICE)
        
        train_data = TensorDataset(X_train_tensor, y_train_tensor)
        test_data = TensorDataset(X_test_tensor, y_test_tensor)
        
        val_start = len(X_train) + SEQUENCE_LENGTH
        val_end = val_start + len(X_test)
        val_time = pd.to_datetime(time_col.iloc[val_start:val_end])
        val_col2_raw = col2_raw[val_start:val_end]
        val_col2_log = col2_log[val_start:val_end]
        
        # 移除emotion_scaler（无情感特征）
        feature_sets_data[feat_set_name] = {
            "train_data": train_data,
            "test_data": test_data,
            "n_feats": n_feats,
            "val_meta": {"time": val_time, "col2_raw": val_col2_raw, "col2_log": val_col2_log},
            "scalers": {"emd": emd_scaler, "target": target_scaler},
            "X_train": X_train, "y_train": y_train,
            "X_test": X_test, "y_test": y_test
        }
    
    return feature_sets_data

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, fc_size, dropout_rate):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            num_layers=1
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_step_out = lstm_out[:, -1, :]
        x = self.dropout(last_step_out)
        out = self.fc(x)
        return out

class LSTMAttentionModel(nn.Module):
    def __init__(self, input_size, hidden_size, attention_size, dropout_rate):
        super(LSTMAttentionModel, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, num_layers=1)
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, attention_size),
            nn.Tanh(),
            nn.Linear(attention_size, 1)
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = self.attention(lstm_out)
        attn_weights = torch.softmax(attn_weights, dim=1)
        context = torch.bmm(lstm_out.transpose(1, 2), attn_weights).squeeze(2)
        x = self.dropout(context)
        out = self.fc(x)
        return out

class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_rate):
        super(RNNModel, self).__init__()
        self.rnn = nn.RNN(
            input_size=input_size, 
            hidden_size=hidden_size, 
            batch_first=True,
            num_layers=2,
            dropout=dropout_rate
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        rnn_out, _ = self.rnn(x)
        last_step_out = rnn_out[:, -1, :]
        x = self.dropout(last_step_out)
        out = self.fc(x)
        return out
        
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_rate):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(
            input_size=input_size, 
            hidden_size=hidden_size, 
            batch_first=True,
            num_layers=2,
            dropout=dropout_rate
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        gru_out, _ = self.gru(x)
        last_step_out = gru_out[:, -1, :]
        x = self.dropout(last_step_out)
        out = self.fc(x)
        return out

class CNNModel(nn.Module):
    def __init__(self, input_size, filters, kernel_size, dropout_rate):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv1d(
            in_channels=input_size,
            out_channels=filters,
            kernel_size=kernel_size,
            padding=1
        )
        self.relu = nn.ReLU()
        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(filters, 1)
    
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.adaptive_pool(x).squeeze(2)
        out = self.fc(x)
        return out

def train_pytorch_model(model, train_loader, test_loader, criterion, optimizer, epochs, device, 
                        patience=EARLY_STOP_PATIENCE, min_delta=EARLY_STOP_MIN_DELTA):
    model.to(device)
    best_test_loss = float('inf')
    counter = 0
    best_model_state = copy.deepcopy(model.state_dict())
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                test_loss += loss.item() * X_batch.size(0)
        
        train_loss /= len(train_loader.dataset)
        test_loss /= len(test_loader.dataset)
        
        if test_loss < best_test_loss - min_delta:
            best_test_loss = test_loss
            best_model_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
        
        if counter >= patience:
            break
    
    model.load_state_dict(best_model_state)
    return best_test_loss, model

def pso_objective_function(params, input_size, train_data, test_data, device):
    hidden_sizes = np.arange(8, 129, 1).tolist()
    fc_sizes = np.arange(8, 65, 1).tolist()
    batch_sizes = np.arange(8, 129, 1).tolist()
    
    hidden_idx = int(np.clip(round(params[0]), 0, len(hidden_sizes)-1))
    fc_idx = int(np.clip(round(params[1]), 0, len(fc_sizes)-1))
    batch_idx = int(np.clip(round(params[2]), 0, len(batch_sizes)-1))
    
    hidden_size = hidden_sizes[hidden_idx]
    fc_size = fc_sizes[fc_idx]
    batch_size = batch_sizes[batch_idx]
    dropout_rate = np.clip(params[3], 0.0, 0.5)
    lr = np.clip(params[4], 1e-5, 1e-3)
    
    if batch_size > len(train_data) or batch_size < 2:
        return 1e10
    
    try:
        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
        
        model = LSTMModel(
            input_size=input_size,
            hidden_size=hidden_size,
            fc_size=fc_size,
            dropout_rate=dropout_rate
        )
        
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        best_test_loss, _ = train_pytorch_model(
            model=model,
            train_loader=train_loader,
            test_loader=test_loader,
            criterion=criterion,
            optimizer=optimizer,
            epochs=200,
            device=device
        )
        return best_test_loss
    
    except Exception:
        return 1e10

def optimize_pso_lstm(input_size, train_data, test_data, device, dataset_name, feat_set_name):
    hidden_sizes = np.arange(8, 129, 1).tolist()
    fc_sizes = np.arange(8, 65, 1).tolist()
    batch_sizes = np.arange(8, 129, 1).tolist()
    
    lb = [0, 0, 0, 0.0, 1e-5]
    ub = [len(hidden_sizes)-1, len(fc_sizes)-1, len(batch_sizes)-1, 0.5, 1e-3]
    
    best_params, best_score = pso(
        func=pso_objective_function,
        lb=lb,
        ub=ub,
        args=(input_size, train_data, test_data, device),
        swarmsize=20,
        maxiter=20,
        debug=False
    )
    
    hidden_idx = int(np.clip(round(best_params[0]), 0, len(hidden_sizes)-1))
    fc_idx = int(np.clip(round(best_params[1]), 0, len(fc_sizes)-1))
    batch_idx = int(np.clip(round(best_params[2]), 0, len(batch_sizes)-1))
    
    best_hidden_size = hidden_sizes[hidden_idx]
    best_fc_size = fc_sizes[fc_idx]
    best_batch_size = batch_sizes[batch_idx]
    best_dropout = best_params[3]
    best_lr = best_params[4]
    
    pso_best_params = {
        "dataset": dataset_name,
        "feature_set": feat_set_name,
        "best_test_loss(MSE)": float(best_score),
        "hyperparameters": {
            "hidden_size": int(best_hidden_size),
            "fc_size": int(best_fc_size),
            "batch_size": int(best_batch_size),
            "dropout_rate": float(best_dropout),
            "learning_rate": float(best_lr)
        }
    }
    
    os.makedirs(f"{SAVE_DIR}/pso_results", exist_ok=True)
    with open(f"{SAVE_DIR}/pso_results/{dataset_name}_{feat_set_name}_PSO_LSTM_best_params.json", "w") as f:
        json.dump(pso_best_params, f, indent=4)
    
    return [best_hidden_size, best_fc_size, best_batch_size, best_dropout, best_lr], best_score

def evaluate_pytorch_model(model, test_loader, scalers, val_meta, device):
    model.eval()
    model.to(device)
    target_scaler = scalers["target"]
    y_true_list = []
    y_pred_list = []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            y_true_list.append(y_batch.cpu().numpy())
            y_pred_list.append(y_pred.cpu().numpy())
    
    y_true_scaled = np.concatenate(y_true_list, axis=0)
    y_pred_scaled = np.concatenate(y_pred_list, axis=0)
    
    target_min = target_scaler.data_min_[0]
    target_max = target_scaler.data_max_[0]
    y_true_log = y_true_scaled * (target_max - target_min) + target_min
    y_pred_log = y_pred_scaled * (target_max - target_min) + target_min
    
    y_true_raw = np.exp(y_true_log)
    y_pred_raw = np.exp(y_pred_log)
    
    metrics = {
        "mae": float(mean_absolute_error(y_true_log, y_pred_log)),
        "mse": float(mean_squared_error(y_true_log, y_pred_log)),
        "rmse": float(np.sqrt(mean_squared_error(y_true_log, y_pred_log))),
        "r2": float(r2_score(y_true_log, y_pred_log)),
        "mape": float(mean_absolute_percentage_error(y_true_log, y_pred_log))
    }
    
    result_data = {
        "metrics": metrics,
        "val_meta": {
            "time": val_meta["time"],
            "col2_raw": val_meta["col2_raw"],
            "col2_log_true": y_true_log.flatten(),
            "col2_log_pred": y_pred_log.flatten(),
            "col2_pred_raw": y_pred_raw.flatten()
        }
    }
    return result_data

def save_prediction_results(dataset_name, feat_set_name, model_name, result_data):
    save_dir = f'{SAVE_DIR}/prediction_results'
    os.makedirs(save_dir, exist_ok=True)
    val_meta = result_data["val_meta"]
    metrics = result_data["metrics"]
    
    min_len = min(len(val_meta["time"]), len(val_meta["col2_raw"]), 
                  len(val_meta["col2_log_true"]), len(val_meta["col2_pred_raw"]))
    
    result_df = pd.DataFrame({
        'date': val_meta["time"].iloc[:min_len].dt.strftime('%Y-%m-%d'),
        'col2_raw': val_meta["col2_raw"][:min_len],
        'col2_log_true': val_meta["col2_log_true"][:min_len],
        'col2_log_pred': val_meta["col2_log_pred"][:min_len],
        'col2_pred_raw': val_meta["col2_pred_raw"][:min_len],
        'log_absolute_error': np.abs(val_meta["col2_log_true"][:min_len] - val_meta["col2_log_pred"][:min_len]),
        'log_relative_error(%)': (np.abs(val_meta["col2_log_true"][:min_len] - val_meta["col2_log_pred"][:min_len]) / 
                                  np.abs(val_meta["col2_log_true"][:min_len])) * 100,
        'raw_absolute_error': np.abs(val_meta["col2_raw"][:min_len] - val_meta["col2_pred_raw"][:min_len]),
        'raw_relative_error(%)': (np.abs(val_meta["col2_raw"][:min_len] - val_meta["col2_pred_raw"][:min_len]) / 
                                  np.abs(val_meta["col2_raw"][:min_len])) * 100
    })
    
    save_path = f'{save_dir}/{dataset_name}_{feat_set_name}_{model_name}_predictions.csv'
    result_df.to_csv(save_path, index=False, encoding='utf-8')
    return metrics

def train_and_evaluate_all_feat_sets(file_path, device):
    feature_sets_data = prepare_data_with_emd_unified_feat(file_path)
    dataset_name = file_path.split("_")[1] if "_" in file_path else "exchange_rate"
    
    common_params = {
        'epochs': 200,     
        'batch_size': 16,
        'dropout': 0.1,
        'lr': 0.0001,
        'units': 24
    }
    
    all_feat_set_results = {}
    overall_best_model = None
    overall_best_metrics = None
    overall_best_name = ""
    
    for feat_set_name, feat_data in feature_sets_data.items():
        train_data = feat_data["train_data"]
        test_data = feat_data["test_data"]
        n_feats = feat_data["n_feats"]
        scalers = feat_data["scalers"]
        val_meta = feat_data["val_meta"]
        input_size = n_feats
        
        base_train_loader = DataLoader(train_data, batch_size=common_params['batch_size'], shuffle=True)
        base_test_loader = DataLoader(test_data, batch_size=common_params['batch_size'], shuffle=False)
        criterion = nn.MSELoss()
        
        base_models = {
            "LSTM": LSTMModel(input_size=input_size, hidden_size=common_params['units'], fc_size=common_params['units'], dropout_rate=common_params['dropout']),
            "LSTM-Attention": LSTMAttentionModel(input_size=input_size, hidden_size=common_params['units'], attention_size=16, dropout_rate=common_params['dropout']),
            "RNN": RNNModel(input_size=input_size, hidden_size=common_params['units'], dropout_rate=common_params['dropout']),
            "GRU": GRUModel(input_size=input_size, hidden_size=common_params['units'], dropout_rate=common_params['dropout']),
            "CNN": CNNModel(input_size=input_size, filters=common_params['units'], kernel_size=3, dropout_rate=common_params['dropout'])
        }
        
        trained_models = {}
        base_model_metrics = {}
        
        for model_name, model in base_models.items():
            optimizer = optim.Adam(model.parameters(), lr=common_params['lr'])
            _, best_model = train_pytorch_model(
                model=model,
                train_loader=base_train_loader,
                test_loader=base_test_loader,
                criterion=criterion,
                optimizer=optimizer,
                epochs=common_params['epochs'],
                device=device
            )
            trained_models[model_name] = best_model
            
            result_data = evaluate_pytorch_model(best_model, base_test_loader, scalers, val_meta, device)
            metrics = save_prediction_results(dataset_name, feat_set_name, model_name, result_data)
            base_model_metrics[model_name] = metrics
            
            if overall_best_metrics is None or metrics['r2'] > overall_best_metrics['r2']:
                overall_best_metrics = metrics
                overall_best_model = best_model
                overall_best_name = f"{model_name}_{feat_set_name}"
        
        best_pso_params, _ = optimize_pso_lstm(input_size, train_data, test_data, device, dataset_name, feat_set_name)
        hidden_size, fc_size, pso_batch_size, dropout_rate, lr = best_pso_params
        
        pso_lstm_model = LSTMModel(
            input_size=input_size,
            hidden_size=int(hidden_size),
            fc_size=int(fc_size),
            dropout_rate=dropout_rate
        )
        
        pso_train_loader = DataLoader(train_data, batch_size=int(pso_batch_size), shuffle=True)
        pso_test_loader = DataLoader(test_data, batch_size=int(pso_batch_size), shuffle=False)
        optimizer = optim.Adam(pso_lstm_model.parameters(), lr=lr)
        
        _, best_pso_model = train_pytorch_model(
            model=pso_lstm_model,
            train_loader=pso_train_loader,
            test_loader=pso_test_loader,
            criterion=criterion,
            optimizer=optimizer,
            epochs=common_params['epochs'],
            device=device
        )
        trained_models["PSO-LSTM"] = best_pso_model
        
        pso_result_data = evaluate_pytorch_model(best_pso_model, pso_test_loader, scalers, val_meta, device)
        pso_metrics = save_prediction_results(dataset_name, feat_set_name, "PSO-LSTM", pso_result_data)
        
        if pso_metrics['r2'] > overall_best_metrics['r2']:
            overall_best_metrics = pso_metrics
            overall_best_model = best_pso_model
            overall_best_name = f"PSO-LSTM_{feat_set_name}"
        
        all_model_metrics = {**base_model_metrics, "PSO-LSTM": pso_metrics}
        all_feat_set_results[feat_set_name] = all_model_metrics
    
    return all_feat_set_results, overall_best_model, overall_best_metrics, overall_best_name

if __name__ == "__main__":
    data_files = ['USDCNY_final.csv']    #Replace dataset
    all_dataset_results = {}
    
    for file in data_files:
        feat_set_results, _, _, _ = train_and_evaluate_all_feat_sets(file, DEVICE)
        dataset_name = file.split("_")[1] if "_" in file else "exchange_rate"
        all_dataset_results[dataset_name] = feat_set_results
    
    for dataset_name, feat_set_dict in all_dataset_results.items():
        for feat_set_name, model_metrics in feat_set_dict.items():
            for model_name, metrics in model_metrics.items():
                print(f"{model_name:<15} MAE={metrics['mae']:.6f} RMSE={metrics['rmse']:.6f} R²={metrics['r2']:.6f} MAPE={metrics['mape']:.6f}")